# 📊 Evaluación exhaustiva de BETO-NER — 4 benchmarks

## Objetivo

Medir el rendimiento real de BETO-NER sobre cuatro benchmarks con características
de dominio y dificultad distintas, para obtener un perfil completo de sus fortalezas
y debilidades antes de usarlo en producción.

---

## Los cuatro benchmarks

| # | Corpus | Fuente | Dominio | Categorías | Entidades | Dificultad |
|---|---|---|---|---|---|---|
| 1 | **CoNLL-2002 ES testb** | EFE (agencia de noticias) | Periodístico | PER/ORG/LOC/MISC | ~3.500 | ★★☆☆ |
| 2 | **WikiANN-ES** | Wikipedia | Enciclopédico | PER/ORG/LOC | ~7.000 | ★★★☆ |
| 3 | **WikiNEuRal-ES** | Wikipedia (BabelNet) | Enciclopédico | PER/ORG/LOC/MISC | ~32.000 | ★★★☆ |
| 4 | **MultiNERD-ES** | Wikipedia + WikiNoticias | Multi-género | PER/ORG/LOC/MISC | ~60.000 | ★★★★ |

### Por qué estos cuatro

**CoNLL-2002** es el benchmark de referencia en NLP en español. BETO-NER puede beneficiarse del texto periodístico estructurado. Sirve como techo de rendimiento conocido.

**WikiANN** expone cómo se comporta el modelo en texto enciclopédico. Sus frases
son más cortas (6–7 tokens de media) y las entidades más variadas
(personajes históricos, topónimos internacionales, organismos poco frecuentes).

**WikiNEuRal** es el sucesor científico de WikiANN, con anotaciones de mayor calidad
generadas por una metodología híbrida neuronal + base de conocimiento (BabelNet).
Tiene cuatro veces más entidades que WikiANN, lo que reduce la varianza estadística.

**MultiNERD** combina Wikipedia y WikiNoticias, los dos géneros textuales más distintos
disponibles en español. Es el benchmark más exigente: la mezcla de géneros obliga
al modelo a generalizar a la vez en estilo informativo y enciclopédico.

---

## Métricas reportadas

| Métrica | Qué mide |
|---|---|
| `Exact-F1` | F1 con coincidencia exacta de span y categoría — métrica principal |
| `Exact-P` | Precisión: % de entidades predichas que son correctas |
| `Exact-R` | Recall: % de entidades reales detectadas |
| `Partial-F1` | F1 con solapamiento ≥ 50% — mide errores de bordes de span |
| `Gap` | Partial − Exact: indica errores de tokenización |
| `F1 por categoría` | PER / ORG / LOC / MISC — dónde falla el modelo |


---

## Metodología estadística: Teorema Central del Límite

Para reportar resultados reproducibles se aplica el **IC95 analítico** basado
en el Teorema Central del Límite, sin coste computacional adicional.

**Fundamento:** cada frase *i* produce un F1ᵢ ∈ [0,1] con su propio TP/FP/FN.
Con *n* frases evaluadas (n ≥ 30 en todos los corpus), el TCL garantiza que
la distribución de la media muestral F̄ es aproximadamente normal:

$$\\bar{F1} \\sim \\mathcal{N}\\!\\left(\\mu,\\, \\frac{\\sigma^2}{n}\\right)$$

El intervalo de confianza al 95% se obtiene directamente:

$$\\text{IC}_{95} = \\bar{F1} \\pm 1.96 \\cdot \\frac{\\sigma}{\\sqrt{n}}$$

**Coste computacional:** cero iteraciones adicionales — usa los *n* valores
F1ᵢ ya calculados en la evaluación principal.

**Reproducibilidad:** dado el mismo corpus y modelo, el IC95 analítico
es determinista — no depende de ninguna semilla aleatoria.


## 0. Instalación

In [1]:
!pip install -q transformers torch datasets pandas numpy matplotlib seaborn tabulate scipy
print("✅ OK")


✅ OK


## 1. Imports, métricas y BETO-NER

In [ ]:
import warnings, time, re, urllib.request
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns


pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_colwidth", 60)

# ── Normalización ─────────────────────────────────────────────
# Mapea cualquier esquema de etiquetado → {PER, ORG, LOC, MISC}
LABEL_MAP = {
    "PER":"PER","PERSON":"PER",
    "B-PER":"PER","I-PER":"PER","S-PER":"PER","E-PER":"PER",
    "ORG":"ORG","NORP":"ORG",
    "B-ORG":"ORG","I-ORG":"ORG","S-ORG":"ORG","E-ORG":"ORG",
    "LOC":"LOC","GPE":"LOC","FAC":"LOC",
    "B-LOC":"LOC","I-LOC":"LOC","S-LOC":"LOC","E-LOC":"LOC",
    "MISC":"MISC","PRODUCT":"MISC","EVENT":"MISC",
    "B-MISC":"MISC","I-MISC":"MISC",
    # MultiNERD: 15 categorías → colapsar en 4
    "ANIM":"MISC","BIO":"MISC","CEL":"MISC","DIS":"MISC",
    "EVE":"MISC","FOOD":"MISC","INST":"MISC","MEDIA":"MISC",
    "MYTH":"MISC","PLANT":"MISC","TIME":"MISC","VEHI":"MISC",
    "B-ANIM":"MISC","B-BIO":"MISC","B-CEL":"MISC","B-DIS":"MISC",
    "B-EVE":"MISC","B-FOOD":"MISC","B-INST":"MISC","B-MEDIA":"MISC",
    "B-MYTH":"MISC","B-PLANT":"MISC","B-TIME":"MISC","B-VEHI":"MISC",
    "I-ANIM":"MISC","I-BIO":"MISC","I-CEL":"MISC","I-DIS":"MISC",
    "I-EVE":"MISC","I-FOOD":"MISC","I-INST":"MISC","I-MEDIA":"MISC",
    "I-MYTH":"MISC","I-PLANT":"MISC","I-TIME":"MISC","I-VEHI":"MISC",
}
def norm(l): return LABEL_MAP.get(l.upper().strip(), "MISC")

# ── Métricas ──────────────────────────────────────────────────
def exact_metrics(gold, pred):
    g = {(s,e,norm(l)) for s,e,l,*_ in gold}
    p = {(s,e,norm(l)) for s,e,l,*_ in pred}
    tp=len(g&p); fp=len(p-g); fn=len(g-p)
    P=tp/(tp+fp) if tp+fp else 0.
    R=tp/(tp+fn) if tp+fn else 0.
    return {"tp":tp,"fp":fp,"fn":fn,"P":P,"R":R,"F1":2*P*R/(P+R) if P+R else 0.}

def partial_metrics(gold, pred, min_j=0.5):
    matched=set(); tp=fp=0
    for ps,pe,pl,*_ in pred:
        pl_n=norm(pl); bj=0; bi=None
        for i,(gs,ge,gl,*_) in enumerate(gold):
            if norm(gl)!=pl_n or i in matched: continue
            inter=max(0,min(pe,ge)-max(ps,gs))
            union=max(pe,ge)-min(ps,gs)
            j=inter/union if union else 0
            if j>bj: bj,bi=j,i
        if bj>=min_j and bi is not None: tp+=1; matched.add(bi)
        else: fp+=1
    fn=len(gold)-len(matched)
    P=tp/(tp+fp) if tp+fp else 0.
    R=tp/(tp+fn) if tp+fn else 0.
    return {"tp":tp,"fp":fp,"fn":fn,"P":P,"R":R,"F1":2*P*R/(P+R) if P+R else 0.}

def aggregate(results):
    tp=sum(r["tp"] for r in results)
    fp=sum(r["fp"] for r in results)
    fn=sum(r["fn"] for r in results)
    P=tp/(tp+fp) if tp+fp else 0.
    R=tp/(tp+fn) if tp+fn else 0.
    return {"tp":tp,"fp":fp,"fn":fn,"P":P,"R":R,"F1":2*P*R/(P+R) if P+R else 0.}

def cat_f1(corpus, pred_fn):
    """F1 por categoría sobre el corpus dado."""
    gold_cats = set(norm(l) for s in corpus for _,_,l,*_ in s["entities"])
    cat_r = defaultdict(list)
    for s in corpus:
        preds = pred_fn(s)
        for cat in gold_cats:
            g = [(a,b,c) for a,b,c,*_ in s["entities"] if norm(c)==cat]
            p = [(a,b,c) for a,b,c,*_ in preds      if norm(c)==cat]
            cat_r[cat].append(exact_metrics(g,p))
    return {c: aggregate(v) for c,v in cat_r.items()}

# ── Stanza ────────────────────────────────────────────────────
import torch
from transformers import pipeline as hf_pipeline

_DEVICE = 0 if torch.cuda.is_available() else -1
print(f"⏳ Cargando BETO-NER ({'GPU' if _DEVICE==0 else 'CPU'})...")
print("   Modelo: mrm8488/bert-spanish-cased-finetuned-ner")
print("   Nota: requiere use_fast=False en el tokenizador")

# BETO-NER necesita el tokenizador lento — sin esto los offsets son incorrectos
_PIPE = hf_pipeline(
    "ner",
    model="mrm8488/bert-spanish-cased-finetuned-ner",
    tokenizer=("mrm8488/bert-spanish-cased-finetuned-ner", {"use_fast": False}),
    aggregation_strategy="simple",
    device=_DEVICE,
)

def predict(sample):
    try:
        raw = _PIPE(sample["text"])
        return [(e["start"], e["end"], e["entity_group"], e["word"]) for e in raw]
    except Exception:
        return []

print("✅ BETO-NER listo")

# ════════════════════════════════════════════════════════════════
# IC95 ANALÍTICO — Teorema Central del Límite
# Sin coste computacional adicional: usa los F1 por frase ya calculados
# ════════════════════════════════════════════════════════════════
import numpy as np

def ic95_analitico(f1_por_frase):
    """
    IC95 analítico basado en el TCL.

    Cada frase i produce un F1_i = exact_metrics(gold_i, pred_i)["F1"].
    Con n frases (n >> 30), el TCL garantiza:

        F̄ ~ N(μ, σ²/n)

    Por tanto:
        IC95 = F̄ ± 1.96 · σ/√n    (z-score al 97.5% = 1.96)

    Parámetros
    ----------
    f1_por_frase : list[float]   F1 de cada frase individual

    Retorna
    -------
    dict con mean, std, se (error estándar), ci95_lo, ci95_hi, n
    """
    arr = np.array(f1_por_frase, dtype=float)
    n   = len(arr)
    mu  = float(np.mean(arr))
    sigma = float(np.std(arr, ddof=1))   # desviación típica muestral
    se  = sigma / np.sqrt(n)             # error estándar de la media
    z   = 1.96                           # z al 97.5% → IC95 bilateral
    return {
        "mean":    mu,
        "std":     sigma,
        "se":      se,
        "ci95_lo": mu - z * se,
        "ci95_hi": mu + z * se,
        "n":       n,
    }

print("✅ Setup OK")
print("   IC95 analítico (TCL): F̄ ± 1.96·σ/√n — sin iteraciones extra")


## 2. Carga de los cuatro benchmarks

In [ ]:
"""
CORPUS LOADER — 4 benchmarks NER español.

C1  CoNLL-2002 ES testb  URL GitHub directa                    ✅
C2  WikiANN-ES           load_dataset("wikiann","es")          ✅
C3  WikiNEuRal-ES        load_dataset("Babelscape/wikineural") ✅  ← SUSTITUYE CoNLL-2002 testa
C4  MultiNERD-ES         JSONL directo (evita bug multinerd.py)✅

CAMBIO RESPECTO A VERSIÓN ANTERIOR:
  C3 era CoNLL-2002 testa (mismo dominio que C1 → redundante para el paper).
  Ahora C3 = WikiNEuRal-ES: anotación neuronal híbrida con BabelNet,
  calidad superior a WikiANN, incluye MISC, dominio Wikipedia.
  Permite medir si la mejora de calidad de anotación afecta al F1 observado.
"""

import re, json, random, urllib.request
from collections import Counter
from datasets import load_dataset

import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def iob_to_corpus(words_list, labels_list):
    """Convierte listas paralelas de words/labels a lista de samples."""
    samples = []
    for words, labels in zip(words_list, labels_list):
        text = " ".join(words)
        char_starts = []; cur = 0
        for w in words: char_starts.append(cur); cur += len(w) + 1
        entities = []; i = 0
        while i < len(labels):
            lbl = labels[i]
            if lbl.startswith("B-") or lbl.startswith("S-"):
                etype = lbl[2:]; start = char_starts[i]; j = i + 1
                while j < len(labels) and (
                    labels[j].startswith("I-") or labels[j].startswith("E-")
                ): j += 1
                end = char_starts[j-1] + len(words[j-1])
                span = text[start:end]
                if span.strip(): entities.append((start, end, etype, span))
                i = j
            else: i += 1
        if entities: samples.append({"text": text, "entities": entities})
    return samples

# ════════════════════════════════════════════════════════════════
# C1 — CoNLL-2002 ES testb
# Dominio: periodístico (EFE) · Cats: PER/ORG/LOC/MISC
# ════════════════════════════════════════════════════════════════
print("⬇️  C1: CoNLL-2002 ES testb...")
_URL = ("https://raw.githubusercontent.com/teropa/nlp/master"
        "/resources/corpora/conll2002/esp.testb")
urllib.request.urlretrieve(_URL, "esp_testb.txt")

sentences, words, labels = [], [], []
with open("esp_testb.txt", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("-DOCSTART-"):
            if words:
                sentences.append((list(words), list(labels)))
                words.clear(); labels.clear()
        else:
            parts = line.split(); words.append(parts[0]); labels.append(parts[-1])
if words: sentences.append((words, labels))

w1, l1 = zip(*sentences)
C1 = iob_to_corpus(list(w1), list(l1))
d1 = Counter(norm(l) for s in C1 for _,_,l,*_ in s["entities"])
print(f"  ✅ CoNLL-2002 testb : {len(C1):5d} frases | "
      f"{sum(len(s['entities']) for s in C1):5d} ents | {dict(d1)}")

# ════════════════════════════════════════════════════════════════
# C2 — WikiANN-ES
# Dominio: enciclopédico · Cats: PER/ORG/LOC
# ════════════════════════════════════════════════════════════════
print("⬇️  C2: WikiANN-ES...")
_WANN  = ["O","B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC"]
_REDIR = re.compile(r'^REDIRECCI[ÓO]N\b', re.IGNORECASE)

ds2 = load_dataset("wikiann", "es", split="test", trust_remote_code=False)
w2, l2 = [], []
for ex in ds2:
    toks = list(ex["tokens"])
    tags = [_WANN[t] if isinstance(t, int) else t for t in ex["ner_tags"]]
    w2.append(toks); l2.append(tags)

C2_raw = iob_to_corpus(w2, l2)
C2 = [s for s in C2_raw
      if not _REDIR.match(s["text"].strip()) and len(s["text"].split()) >= 4]
random.shuffle(C2); C2 = C2[:1000]
d2 = Counter(norm(l) for s in C2 for _,_,l,*_ in s["entities"])
print(f"  ✅ WikiANN-ES       : {len(C2):5d} frases | "
      f"{sum(len(s['entities']) for s in C2):5d} ents | {dict(d2)}")

# ════════════════════════════════════════════════════════════════
# C3 — WikiNEuRal-ES  ← NUEVO (sustituye CoNLL-2002 testa)
# Dominio: Wikipedia con anotación neuronal + BabelNet
# Cats: PER/ORG/LOC/MISC · Calidad anotación > WikiANN
# Por qué: mide si la calidad de anotación afecta al F1 observado
#           comparado con WikiANN (mismo dominio, distinta calidad)
#
# FIX: Babelscape/wikineural solo tiene config "default" (multilingüe).
# No acepta "es" como nombre de config — hay que cargar "default"
# y filtrar por el campo "langs" == "es".
# ════════════════════════════════════════════════════════════════
print("⬇️  C3: WikiNEuRal-ES (Babelscape/wikineural, filtrado langs=es)...")
_WNEUR_ID2L = {
    0:"O", 1:"B-PER", 2:"I-PER",
    3:"B-ORG",  4:"I-ORG",
    5:"B-LOC",  6:"I-LOC",
    7:"B-MISC", 8:"I-MISC",
}

# Split "test_es" contiene directamente las frases en español
# (el error "Unknown split test" confirma que los splits son por idioma)
ds3 = load_dataset("Babelscape/wikineural",
                   split="test_es", trust_remote_code=False)
w3, l3 = [], []
for ex in ds3:
    toks = list(ex["tokens"])
    tags = [_WNEUR_ID2L.get(t, "O") if isinstance(t, int) else t
            for t in ex["ner_tags"]]
    w3.append(toks); l3.append(tags)
C3_raw = iob_to_corpus(w3, l3)
random.shuffle(C3_raw); C3 = C3_raw[:1000]
d3 = Counter(norm(l) for s in C3 for _,_,l,*_ in s["entities"])
print(f"  ✅ WikiNEuRal-ES    : {len(C3):5d} frases | "
      f"{sum(len(s['entities']) for s in C3):5d} ents | {dict(d3)}")

# ════════════════════════════════════════════════════════════════
# C4 — MultiNERD-ES  (Wikipedia + WikiNoticias, 15 categorías)
# FIX: JSONL directo para evitar RuntimeError de multinerd.py
# ════════════════════════════════════════════════════════════════
print("⬇️  C4: MultiNERD-ES (JSONL directo)...")
_MNERD_ID2L = {
    0:"O", 1:"B-PER",2:"I-PER", 3:"B-ORG",4:"I-ORG", 5:"B-LOC",6:"I-LOC",
    7:"B-ANIM",8:"I-ANIM", 9:"B-BIO",10:"I-BIO", 11:"B-CEL",12:"I-CEL",
    13:"B-DIS",14:"I-DIS", 15:"B-EVE",16:"I-EVE", 17:"B-FOOD",18:"I-FOOD",
    19:"B-INST",20:"I-INST", 21:"B-MEDIA",22:"I-MEDIA",
    23:"B-MYTH",24:"I-MYTH", 25:"B-PLANT",26:"I-PLANT",
    27:"B-TIME",28:"I-TIME", 29:"B-VEHI",30:"I-VEHI",
}
_MNERD_URL = ("https://huggingface.co/datasets/tner/multinerd"
              "/resolve/main/dataset/es.jsonl")
print("    Descargando es.jsonl (~30 MB)...")
urllib.request.urlretrieve(_MNERD_URL, "multinerd_es.jsonl")

w4, l4 = [], []
with open("multinerd_es.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        obj = json.loads(line)
        toks = obj["tokens"]
        tags = [
            _MNERD_ID2L.get(int(t), "O") if str(t).lstrip("-").isdigit() else str(t)
            for t in obj["tags"]
        ]
        w4.append(toks); l4.append(tags)

C4_raw = iob_to_corpus(w4, l4)
random.shuffle(C4_raw); C4 = C4_raw[:1000]
d4 = Counter(norm(l) for s in C4 for _,_,l,*_ in s["entities"])
print(f"  ✅ MultiNERD-ES     : {len(C4):5d} frases | "
      f"{sum(len(s['entities']) for s in C4):5d} ents | {dict(d4)}")

CORPORA = {
    "CoNLL-2002":  (C1, "Periodístico EFE",                    "★★☆☆"),
    "WikiANN":     (C2, "Enciclopédico (Wikipedia)",            "★★★☆"),
    "WikiNEuRal":  (C3, "Enciclopédico (Wikipedia+BabelNet)",   "★★★☆"),
    "MultiNERD":   (C4, "Multi-género (Wiki+WikiNoticias)",     "★★★★"),
}
print(f"\n✅ {len(CORPORA)} benchmarks listos — semilla={RANDOM_SEED}")


## 3. Evaluación global

In [ ]:
print("⏳ Evaluando BETO-NER...")
print("   IC95 analítico (TCL): sin iteraciones extra\n")

RESULTS = []
CAT_RESULTS = {}
ERROR_SAMPLES = {}
IC95_RESULTS = {}   # {cname: dict IC95}

for cname, (corpus, desc, diff) in CORPORA.items():
    if not corpus:
        print(f"  ⚠️  {cname}: corpus vacío — omitido")
        continue
    t0 = time.time()
    ex_r, par_r = [], []
    f1_por_frase = []   # F1 individual de cada frase → para IC95 analítico
    fps, fns = [], []

    for s in corpus:
        preds = predict(s)
        m_ex  = exact_metrics(s["entities"], preds)
        m_par = partial_metrics(s["entities"], preds)
        ex_r.append(m_ex)
        par_r.append(m_par)
        f1_por_frase.append(m_ex["F1"])   # F1_i de cada frase

        gold_set = {(gs,ge,norm(gl)) for gs,ge,gl,*_ in s["entities"]}
        pred_set = {(ps,pe,norm(pl)) for ps,pe,pl,*_ in preds}
        pred_map = {(ps,pe,norm(pl)):sp for ps,pe,pl,sp in preds}
        gold_map = {(gs,ge,norm(gl)):gt for gs,ge,gl,*rest in s["entities"]
                    for gt in [rest[0] if rest else s["text"][gs:ge]]}
        for key in pred_set - gold_set:
            ctx = s["text"][max(0,key[0]-25):key[1]+25].replace("\n"," ")
            fps.append({"cat":key[2],"span":pred_map.get(key,""),"ctx":ctx})
        for key in gold_set - pred_set:
            ctx = s["text"][max(0,key[0]-25):key[1]+25].replace("\n"," ")
            fns.append({"cat":key[2],"span":gold_map.get(key,""),"ctx":ctx})

    ex  = aggregate(ex_r)
    par = aggregate(par_r)
    ms  = (time.time()-t0)*1000/len(corpus)

    # IC95 analítico (TCL) — coste cero, usa f1_por_frase ya calculados
    ic = ic95_analitico(f1_por_frase)
    IC95_RESULTS[cname] = ic

    CAT_RESULTS[cname] = cat_f1(corpus, predict)
    ERROR_SAMPLES[cname] = {"fp": fps, "fn": fns}

    RESULTS.append({
        "Benchmark":  cname,
        "Dominio":    desc,
        "Dificultad": diff,
        "n_frases":   len(corpus),
        "Entidades":  sum(len(s["entities"]) for s in corpus),
        "Exact-F1":   ex["F1"],
        "F1_mean":    ic["mean"],
        "F1_std":     ic["std"],
        "F1_se":      ic["se"],
        "IC95_lo":    ic["ci95_lo"],
        "IC95_hi":    ic["ci95_hi"],
        "Exact-P":    ex["P"],
        "Exact-R":    ex["R"],
        "Partial-F1": par["F1"],
        "Gap":        round(par["F1"]-ex["F1"],4),
        "FP":         ex["fp"],
        "FN":         ex["fn"],
        "TP":         ex["tp"],
        "ms/frase":   round(ms,1),
    })

    # Verificar TCL: n >> 30 ✅
    tcl_ok = "✅ TCL válido" if ic["n"] >= 30 else "⚠️  n<30, TCL no garantizado"
    print(f"  {cname:14s}  n={ic['n']:5d}  {tcl_ok}")
    print(f"    F1={ex['F1']:.4f}  μ={ic['mean']:.4f} ± σ={ic['std']:.4f}"
          f"  SE={ic['se']:.4f}  IC95=[{ic['ci95_lo']:.4f}, {ic['ci95_hi']:.4f}]"
          f"  FP={ex['fp']}  FN={ex['fn']}  {ms:.0f}ms/frase")

df = pd.DataFrame(RESULTS)
print("\n✅ Evaluación completa")


## 4. Tabla de resultados

In [ ]:
SEP = "="*120
print(f"\n{SEP}")
print("  BETO-NER — IC95 ANALÍTICO (TCL): F̄ ± 1.96·σ/√n")
print(f"{SEP}")
try:
    from tabulate import tabulate
    show = ["Benchmark","Dificultad","n_frases",
            "F1_mean","F1_std","F1_se","IC95_lo","IC95_hi",
            "Exact-P","Exact-R","Partial-F1","Gap","FP","FN"]
    disp = df[show].copy()
    for c in ["F1_mean","F1_std","F1_se","IC95_lo","IC95_hi",
              "Exact-P","Exact-R","Partial-F1","Gap"]:
        disp[c] = disp[c].map(lambda x: f"{x:.4f}")
    print(tabulate(disp, headers="keys", tablefmt="fancy_grid", showindex=False))
except ImportError:
    print(df[["Benchmark","F1_mean","F1_std","IC95_lo","IC95_hi",
              "Exact-P","Exact-R","FP","FN"]].to_string(index=False))
print(f"{SEP}")

print("\n── Interpretación estadística ──")
print("  F1_mean = media de los F1 individuales por frase (= F1 global por TCL)")
print("  F1_std  = desviación típica entre frases (σ del rendimiento por frase)")
print("  F1_se   = error estándar de la media = σ/√n")
print("  IC95    = F̄ ± 1.96·SE  (intervalo al 95% por TCL, z=1.96)")
print()
for _, row in df.iterrows():
    amplitud = row["IC95_hi"] - row["IC95_lo"]
    estab = "estable" if row["F1_std"] < 0.30 else "variable"
    tcl = f"n={int(row['n_frases'])} >> 30 ✅" if row["n_frases"] >= 30 else "⚠️  n<30"
    print(f"  {row['Benchmark']:14s}  F1={row['F1_mean']:.4f} ± {row['F1_std']:.4f}"
          f"  IC95=[{row['IC95_lo']:.4f}, {row['IC95_hi']:.4f}]"
          f"  amplitud={amplitud:.4f}  {tcl}")

rng_f1 = df["F1_mean"].max() - df["F1_mean"].min()
best  = df.loc[df["F1_mean"].idxmax(), "Benchmark"]
worst = df.loc[df["F1_mean"].idxmin(), "Benchmark"]
print(f"\n  Mejor: {best} (F1={df['F1_mean'].max():.4f})")
print(f"  Peor:  {worst} (F1={df['F1_mean'].min():.4f})")
print(f"  Rango: {rng_f1:.4f} puntos F1")


## 5. F1 por categoría en cada benchmark

Esta sección revela dónde falla exactamente Stanza en cada dominio.
Un F1 bajo en una categoría específica indica que el modelo tiene
dificultades con ese tipo de entidad en ese contexto.


In [ ]:
cat_rows = []
for cname, cat_res in CAT_RESULTS.items():
    for cat, m in cat_res.items():
        cat_rows.append({
            "Benchmark": cname, "Categoría": cat,
            "F1": m["F1"], "P": m["P"], "R": m["R"],
            "FP": m["fp"], "FN": m["fn"], "TP": m["tp"]
        })
df_cat = pd.DataFrame(cat_rows)

# Tabla
print("── F1 por categoría y benchmark ──")
pivot = df_cat.pivot_table(index="Benchmark", columns="Categoría", values="F1")
print(pivot.round(4).to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(8, 4))
mask = pivot.isnull()
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.6, ax=ax,
            mask=mask, cbar_kws={"label": "F1 exacto"},
            annot_kws={"size": 11})
ax.set_title("BETO-NER — F1 por categoría y benchmark",
             fontweight="bold", pad=12, fontsize=12)
ax.set_xlabel("Categoría NER"); ax.set_ylabel("")
plt.tight_layout()
plt.savefig("beto_evaluacion_4benchmarks_v2_heatmap_cat.png", dpi=150, bbox_inches="tight")
plt.show()

# Diagnóstico rápido
print("\n── Diagnóstico por categoría ──")
for cat in ["PER","ORG","LOC","MISC"]:
    sub = df_cat[df_cat["Categoría"]==cat]
    if sub.empty: continue
    mean_f1 = sub["F1"].mean()
    std_f1  = sub["F1"].std()
    worst_b = sub.loc[sub["F1"].idxmin(), "Benchmark"]
    print(f"  {cat:4s}  media={mean_f1:.3f}  std={std_f1:.3f}  "
          f"peor benchmark: {worst_b} (F1={sub['F1'].min():.3f})")


## 6. Visualizaciones

In [ ]:
BENCH_COLORS = {
    "CoNLL-2002": "#185FA5",
    "WikiANN":          "#BA7517",
    "MultiNERD":        "#534AB7",
    "WikiNEuRal":       "#BB8852"
}

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("BETO-NER — evaluación sobre 4 benchmarks",
             fontsize=13, fontweight="bold", y=1.01)

# ── Gráfico 1: F1 por benchmark ──────────────────────────────
ax = axes[0,0]
colors = [BENCH_COLORS[b] for b in df["Benchmark"]]
bars = ax.bar(df["Benchmark"], df["Exact-F1"],
              color=colors, edgecolor="white", width=0.55)
for bar, v, pf in zip(bars, df["Exact-F1"], df["Partial-F1"]):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.008,
            f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")
    ax.text(bar.get_x()+bar.get_width()/2, v-0.035,
            f"(partial {pf:.3f})", ha="center", fontsize=8, color="#444")
ax.set_ylim(0, 1.05)
ax.set_ylabel("F1 exacto"); ax.set_title("F1 exacto por benchmark", fontweight="bold")
ax.axhline(df["Exact-F1"].mean(), color="gray", ls="--", lw=1.2,
           label=f"media={df['Exact-F1'].mean():.3f}")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

# ── Gráfico 2: Precisión vs Recall ───────────────────────────
ax2 = axes[0,1]
for _, r in df.iterrows():
    c = BENCH_COLORS.get(r["Benchmark"],"#888")
    ax2.scatter(r["Exact-R"], r["Exact-P"], color=c, s=180,
                zorder=3, edgecolor="white", linewidth=1)
    ax2.annotate(r["Benchmark"],
                 xy=(r["Exact-R"], r["Exact-P"]),
                 xytext=(5, 4), textcoords="offset points", fontsize=9)
# Curvas iso-F1
for f1_val in [0.5, 0.6, 0.7, 0.8, 0.9]:
    p_vals = np.linspace(0.3, 1.0, 300)
    r_vals = f1_val*p_vals/(2*p_vals-f1_val)
    mask = (r_vals>=0.3) & (r_vals<=1.0)
    ax2.plot(r_vals[mask], p_vals[mask], "--", color="gray", alpha=0.25, lw=0.8)
    if mask.any():
        ax2.text(r_vals[mask][-1]+0.01, p_vals[mask][-1],
                 f"F1={f1_val:.1f}", fontsize=7.5, color="gray")
ax2.set_xlabel("Recall"); ax2.set_ylabel("Precisión")
ax2.set_xlim(0.3, 1.05); ax2.set_ylim(0.3, 1.05)
ax2.set_title("Precisión vs Recall\n(con curvas iso-F1)", fontweight="bold")
ax2.grid(alpha=0.3)

# ── Gráfico 3: F1 por categoría (grouped bar) ────────────────
ax3 = axes[1,0]
cats_avail = sorted(set(df_cat["Categoría"].unique()) & {"PER","ORG","LOC","MISC"})
benchmarks = list(CORPORA.keys())
x = np.arange(len(cats_avail)); w = 0.2
for i, bname in enumerate(benchmarks):
    sub = df_cat[df_cat["Benchmark"]==bname]
    vals = [sub[sub["Categoría"]==c]["F1"].values[0]
            if c in sub["Categoría"].values else 0
            for c in cats_avail]
    bars3 = ax3.bar(x + i*w, vals, w,
                    label=bname, color=BENCH_COLORS[bname],
                    edgecolor="white", alpha=0.88)
ax3.set_xticks(x + w*1.5); ax3.set_xticklabels(cats_avail, fontsize=11)
ax3.set_ylabel("F1 exacto"); ax3.set_title("F1 por categoría", fontweight="bold")
ax3.set_ylim(0, 1.05); ax3.legend(fontsize=9); ax3.grid(axis="y", alpha=0.3)

# ── Gráfico 4: FP y FN por benchmark ─────────────────────────
ax4 = axes[1,1]
x4 = np.arange(len(df)); w4 = 0.35
ax4.bar(x4-w4/2, df["FP"], w4, label="FP (pred. incorrectas)",
        color="#C84B2F", edgecolor="white", alpha=0.88)
ax4.bar(x4+w4/2, df["FN"], w4, label="FN (entidades perdidas)",
        color="#185FA5", edgecolor="white", alpha=0.88)
for i, (fp, fn) in enumerate(zip(df["FP"], df["FN"])):
    ax4.text(i-w4/2, fp+15, str(fp), ha="center", fontsize=9)
    ax4.text(i+w4/2, fn+15, str(fn), ha="center", fontsize=9)
ax4.set_xticks(x4); ax4.set_xticklabels(df["Benchmark"])
ax4.set_ylabel("Número de errores")
ax4.set_title("Falsos positivos y falsos negativos\npor benchmark", fontweight="bold")
ax4.legend(fontsize=9); ax4.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("beto_evaluacion_4benchmarks_v2.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Gráfico extra: IC95 con barras de error ───────────────────
fig2, ax2 = plt.subplots(figsize=(8, 4))
benchmarks = df["Benchmark"].tolist()
means      = df["F1_mean"].tolist()
errors     = (1.96 * df["F1_se"]).tolist()   # semiancho del IC95
colors_ic  = [BENCH_COLORS.get(b,"#888") for b in benchmarks]

bars = ax2.bar(benchmarks, means, color=colors_ic,
               edgecolor="white", alpha=0.88, width=0.55)
ax2.errorbar(benchmarks, means, yerr=errors,
             fmt="none", color="black", capsize=6, linewidth=1.5, capthick=1.5)

for bar, m, lo, hi in zip(bars, means, df["IC95_lo"], df["IC95_hi"]):
    ax2.text(bar.get_x()+bar.get_width()/2, m+max(errors)*1.3,
             f"{m:.4f}\n[{lo:.4f},{hi:.4f}]",
             ha="center", fontsize=8, linespacing=1.4)

ax2.set_ylabel("F1 exacto (media ± IC95)")
ax2.set_title("BETO-NER — F1 con IC95 analítico (TCL: F̄ ± 1.96·σ/√n)",
              fontweight="bold")
ax2.set_ylim(0, min(1.0, max(means)+0.15))
ax2.grid(axis="y", alpha=0.3)
# Nota metodológica
ax2.text(0.01, 0.02,
         "IC95 analítico por TCL. Cada frase produce un F1ᵢ; n>>30 garantiza normalidad de F̄.",
         transform=ax2.transAxes, fontsize=7.5, color="#555",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
plt.tight_layout()
plt.savefig("beto_evaluacion_4benchmarks_v2_ic95.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Gráfico IC95 guardado")


## 7. Convergencia del TCL — estabilización del error estándar

El Teorema Central del Límite garantiza que, **a medida que aumenta el número
de frases evaluadas $n$**, el error estándar $SE_n = \sigma / \sqrt{n}$ decrece
y el intervalo de confianza se estrecha hasta estabilizarse.

Esta celda visualiza ese proceso de estabilización para cada corpus:

- **Eje X**: número de frases acumuladas (de 1 a n total)
- **Eje Y izquierdo**: $\bar{F}_1$ acumulado — cómo converge la media
- **Eje Y derecho**: semiancho del IC95 ($1.96 \cdot SE_n$) — cómo se estrecha el intervalo
- **Línea vertical gris**: umbral $n = 30$ a partir del cual el TCL es aplicable
- **Banda gris**: IC95 final estabilizado


In [ ]:
print("⏳ Calculando convergencia del TCL por corpus...\n")

BENCH_COLORS = {
    "CoNLL-2002": "#185FA5",
    "WikiANN":    "#BA7517",
    "WikiNEuRal": "#0F6E56",
    "MultiNERD":  "#534AB7",
}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(
    "Convergencia del TCL — estabilización del IC95\n"
    "beto",
    fontsize=13, fontweight="bold"
)

for ax, (cname, (corpus, desc, diff)) in zip(axes.flat, CORPORA.items()):
    color = BENCH_COLORS.get(cname, "#888")

    # ── Calcular f1_por_frase en orden de evaluación ──────────
    f1_seq = [exact_metrics(s["entities"], predict(s))["F1"]
              for s in corpus]
    n_total = len(f1_seq)

    # ── Estadísticos acumulados para cada n ──────────────────
    ns         = np.arange(1, n_total + 1)
    f1_arr     = np.array(f1_seq)
    mean_cum   = np.cumsum(f1_arr) / ns                  # F̄₁ acumulado
    # Varianza acumulada con corrección de Bessel (ddof=1)
    # Usamos la fórmula incremental: Var_n = (Σx² - n·μ²) / (n-1)
    sq_cum     = np.cumsum(f1_arr**2)
    var_cum    = np.where(ns > 1,
                          (sq_cum - ns * mean_cum**2) / (ns - 1),
                          0.0)
    var_cum    = np.maximum(var_cum, 0.0)   # evitar negativos numéricos
    se_cum     = np.sqrt(var_cum / ns)      # SE acumulado = σ_n / √n
    semi_ic95  = 1.96 * se_cum              # semiancho IC95

    # ── Valores finales estabilizados ─────────────────────────
    f1_final   = mean_cum[-1]
    se_final   = se_cum[-1]
    ic_lo      = f1_final - 1.96 * se_final
    ic_hi      = f1_final + 1.96 * se_final

    # ── Ejes gemelos ──────────────────────────────────────────
    ax2 = ax.twinx()

    # Banda IC95 final (referencia de convergencia)
    ax.axhspan(ic_lo, ic_hi, color=color, alpha=0.08, label=f"IC95 final")

    # Curva F̄₁ acumulado
    ax.plot(ns, mean_cum, color=color, lw=1.8, label=f"$\\bar{{F}}_1$ acumulado")
    ax.axhline(f1_final, color=color, lw=0.8, ls="--", alpha=0.6)

    # Curva semiancho IC95 en eje derecho
    ax2.plot(ns, semi_ic95, color=color, lw=1.4, ls=":", alpha=0.85,
             label="1.96·SE (semiancho IC95)")
    ax2.axhline(1.96 * se_final, color=color, lw=0.8, ls=":", alpha=0.5)
    ax2.set_ylabel("1.96·SE (semiancho IC95)", fontsize=8, color="#555")
    ax2.tick_params(axis="y", labelsize=8, colors="#555")
    ax2.set_ylim(bottom=0)

    # Línea vertical: umbral TCL válido (n=30)
    ax.axvline(30, color="gray", lw=1.0, ls="-.", alpha=0.7, label="n=30 (TCL válido)")

    # Anotación del valor final
    ax.annotate(
        f"$\\bar{{F}}_1$ = {f1_final:.4f}\n"
        f"SE = {se_final:.4f}\n"
        f"IC95 = [{ic_lo:.4f}, {ic_hi:.4f}]\n"
        f"Amplitud = {ic_hi-ic_lo:.4f}",
        xy=(n_total, f1_final),
        xytext=(max(30, n_total*0.55), f1_final - 0.12),
        fontsize=8, color=color,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor=color, alpha=0.9),
        arrowprops=dict(arrowstyle="->", color=color, lw=0.8)
    )

    ax.set_xlabel("Frases evaluadas (n acumulado)", fontsize=9)
    ax.set_ylabel("$\\bar{F}_1$ acumulado", fontsize=9)
    ax.set_title(f"{cname}  ({desc})", fontweight="bold", fontsize=10)
    ax.set_xlim(1, n_total)
    ax.set_ylim(
        max(0, f1_final - 0.25),
        min(1.02, f1_final + 0.25)
    )
    ax.grid(alpha=0.25)

    # Leyenda combinada (eje izquierdo + derecho)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7.5, loc="upper right")

plt.tight_layout()
plt.savefig("BETO.pdf", bbox_inches="tight")
plt.show()
print("✅ Gráfico guardado: BETO.pdf")

# ── Tabla resumen de convergencia ─────────────────────────────
print()
print("═"*75)
print("  Resumen de convergencia del TCL por corpus")
print("═"*75)
print(f"  {'Corpus':14s}  {'n':>5s}  {'F̄₁':>8s}  {'σ':>8s}  "
      f"{'SE':>8s}  {'IC95':^24s}  {'Amplitud':>9s}")
print("─"*75)
for cname, (corpus, desc, diff) in CORPORA.items():
    f1_seq  = [exact_metrics(s["entities"], predict(s))["F1"] for s in corpus]
    arr     = np.array(f1_seq)
    n       = len(arr)
    mu      = arr.mean()
    sigma   = arr.std(ddof=1)
    se      = sigma / np.sqrt(n)
    lo, hi  = mu - 1.96*se, mu + 1.96*se
    print(f"  {cname:14s}  {n:>5d}  {mu:>8.4f}  {sigma:>8.4f}  "
          f"{se:>8.4f}  [{lo:.4f}, {hi:.4f}]  {hi-lo:>9.4f}")
print("═"*75)
print()
print("  Interpretación:")
print("  · SE = σ/√n — se reduce con √n: duplicar n reduce SE a la mitad")
print("  · Amplitud IC95 = 2·1.96·SE — cuanto más pequeña, más precisa la estimación")
print("  · La curva de convergencia muestra que a partir de n≈100-200")
print("    el IC95 ya está prácticamente estabilizado en todos los corpus")


## 7. Análisis cualitativo de errores

Para cada benchmark mostramos los FP y FN más frecuentes, clasificados por categoría
y con contexto. Esto permite entender **por qué** falla el modelo, no solo cuánto.


In [ ]:
# Taxonomía de patrones de error
def classify_error(span, cat, ctx, is_fp):
    s = span.lower().strip() if span else ""
    ctx_l = ctx.lower()
    if is_fp:
        # Patrones FP comunes
        if cat == "MISC" and len(s.split()) == 1 and s.istitle():
            return "FP: palabra capitalizada genérica"
        if cat == "LOC" and any(kw in ctx_l for kw in
                                 ["selección","jugadores","equipo","partido"]):
            return "FP: país en contexto deportivo (debería ser ORG)"
        if cat == "PER" and any(c.isdigit() for c in s):
            return "FP: entidad con números clasificada como PER"
        return f"FP: {cat} incorrecto"
    else:
        # Patrones FN comunes
        if any(q in ctx for q in ["'","'",'"','"','«','»']):
            return "FN: entidad entre comillas no detectada"
        if len(span.split()) >= 4:
            return "FN: entidad larga no detectada"
        return f"FN: {cat} no detectado"

print("═"*70)
for cname, errs in ERROR_SAMPLES.items():
    fps = errs["fp"]; fns = errs["fn"]
    print(f"\n{'─'*70}")
    print(f"  {cname}  —  {len(fps)} FP | {len(fns)} FN")
    print(f"{'─'*70}")
    print(f"  FP por categoría: {dict(Counter(r['cat'] for r in fps))}")
    print(f"  FN por categoría: {dict(Counter(r['cat'] for r in fns))}")

    # Patrones más frecuentes
    fp_pats = Counter(classify_error(r["span"],r["cat"],r["ctx"],True) for r in fps)
    fn_pats = Counter(classify_error(r["span"],r["cat"],r["ctx"],False) for r in fns)
    print(f"\n  Patrones FP más frecuentes:")
    for pat,n in fp_pats.most_common(4):
        print(f"    [{n:4d}] {pat}")
    print(f"  Patrones FN más frecuentes:")
    for pat,n in fn_pats.most_common(4):
        print(f"    [{n:4d}] {pat}")

    print(f"\n  Ejemplos FP ({cname}):")
    for r in fps[:5]:
        print(f"    [{r['cat']:4s}] '{r['span'][:35]:35s}' | {r['ctx'][:55]}")
    print(f"  Ejemplos FN ({cname}):")
    for r in fns[:5]:
        print(f"    [{r['cat']:4s}] '{r['span'][:35]:35s}' | {r['ctx'][:55]}")
print("\n" + "═"*70)


## 8. Diagnóstico final y exportación

In [ ]:
# ── Tabla resumen ─────────────────────────────────────────────
print("\n" + "="*75)
print("  DIAGNÓSTICO FINAL — BETO-NER")
print("="*75)

for _, row in df.iterrows():
    f1   = row["Exact-F1"]
    label = ("🟢 Muy bueno" if f1 >= 0.85
             else "🟡 Aceptable" if f1 >= 0.70
             else "🟠 Débil"    if f1 >= 0.55
             else "🔴 Insuficiente")
    print(f"  {row['Benchmark']:12s} {row['Dificultad']}  "
          f"F1={f1:.4f}  {label}")

print()
print("── Conclusiones ──")
best_c  = df.loc[df["Exact-F1"].idxmax()]
worst_c = df.loc[df["Exact-F1"].idxmin()]
gap     = best_c["Exact-F1"] - worst_c["Exact-F1"]

print(f"  • Mejor resultado en {best_c['Benchmark']} (F1={best_c['Exact-F1']:.4f}): "
      f"dominio periodístico cercano al corpus de entrenamiento.")
print(f"  • Peor resultado en {worst_c['Benchmark']} (F1={worst_c['Exact-F1']:.4f}): "
      f"dominio más alejado del entrenamiento.")
print(f"  • Brecha entre mejor y peor benchmark: {gap:.4f} puntos F1.")
if gap > 0.20:
    print(f"    → Brecha ALTA (>{0.20:.2f}): Stanza generaliza mal entre dominios.")
elif gap > 0.10:
    print(f"    → Brecha MEDIA ({gap:.2f}): generalización moderada.")
else:
    print(f"    → Brecha BAJA ({gap:.2f}): Stanza generaliza bien entre dominios.")

# ── PER: categoría más estable ────────────────────────────────
per_vals = df_cat[df_cat["Categoría"]=="PER"]["F1"]
org_vals = df_cat[df_cat["Categoría"]=="ORG"]["F1"]
loc_vals = df_cat[df_cat["Categoría"]=="LOC"]["F1"]
misc_vals = df_cat[df_cat["Categoría"]=="MISC"]["F1"]

print()
print("── Rendimiento por categoría (media ± std entre benchmarks) ──")
for cat, vals in [("PER",per_vals),("ORG",org_vals),
                   ("LOC",loc_vals),("MISC",misc_vals)]:
    if vals.empty: continue
    print(f"  {cat:4s}  media={vals.mean():.3f}  std={vals.std():.3f}  "
          f"min={vals.min():.3f}  max={vals.max():.3f}")

# ── Exportar ──────────────────────────────────────────────────
df.to_csv("beto_evaluacion_4benchmarks_v2_benchmarks.csv", index=False, float_format="%.4f")
df_cat.to_csv("beto_evaluacion_4benchmarks_v2_categorias.csv", index=False, float_format="%.4f")
print("\n✅ Exportado: stanza_evaluacion_benchmarks.csv · stanza_evaluacion_categorias.csv")

# Tabla LaTeX
latex = df[["Benchmark","Exact-F1","Exact-P","Exact-R",
            "Partial-F1","Gap","FP","FN"]].to_latex(
    index=False, float_format="%.4f", column_format="lrrrrrcc",
    caption="Evaluación de BETO-NER sobre cuatro benchmarks.",
    label="tab:stanza_eval", escape=False)
with open("beto_evaluacion_4benchmarks_v2_tab.tex","w") as f: f.write(latex)
print("   tab_stanza_evaluacion.tex")

# ── PER: categoría más estable ────────────────────────────────
print("\n── Estabilidad estadística (bootstrap) ──")
print(f"  {'Benchmark':14s}  {'F1_mean':>8s}  {'F1_std':>8s}  {'IC95':>22s}  Amplitud")
for _, row in df.iterrows():
    amplitud = row["IC95_hi"] - row["IC95_lo"]
    print(f"  {row['Benchmark']:14s}  {row['F1_mean']:8.4f}  {row['F1_std']:8.4f}  "
          f"[{row['IC95_lo']:.4f}, {row['IC95_hi']:.4f}]  {amplitud:.4f}")

# Exportar con columnas estadísticas completas
df.to_csv("beto_evaluacion_4benchmarks_v2_benchmarks.csv", index=False, float_format="%.4f")
df_cat.to_csv("beto_evaluacion_4benchmarks_v2_categorias.csv", index=False, float_format="%.4f")

# LaTeX listo para paper — formato estándar NLP
latex = df[["Benchmark","n_frases","F1_mean","F1_std","F1_se","IC95_lo","IC95_hi",
            "Exact-P","Exact-R","Partial-F1","FP","FN"]].to_latex(
    index=False, float_format="%.4f",
    column_format="lcrrrrrrrcc",
    caption=("Evaluación de BETO-NER sobre cuatro benchmarks. "
             "F1 reportado como $\\bar{F1} \\pm \\sigma$ donde $\\bar{F1}$ es la media "
             "de los F1 por frase, $\\sigma$ la desviación típica muestral y "
             "el IC95 se obtiene por el TCL: $\\bar{F1} \\pm 1.96 \\cdot \\sigma/\\sqrt{n}$."),
    label="tab:spacy_lg_eval",
    escape=False,
)
with open("beto_evaluacion_4benchmarks_v2_tab.tex","w") as f: f.write(latex)
print("\n✅ Exportado: spacy_lg_evaluacion_benchmarks.csv, .tex")
print("   IC95 analítico — determinista, sin semilla aleatoria")


## Conclusiones

### Interpretación del perfil de BETO-NER

Los cuatro benchmarks revelan aspectos distintos del modelo que un benchmark único
nunca mostraría:

**CoNLL-2002** mide el rendimiento en el dominio más cercano al entrenamiento.
BETO-NER fue entrenado en BETO (BERT español, U. Chile) → fine-tuned en CoNLL-2002 ES, corpus periodístico español, y CoNLL-2002 proviene
de la agencia EFE. El resultado es el más alto, pero puede dar una imagen demasiado
optimista para texto real.

**WikiANN** expone la robustez en texto enciclopédico corto. Las frases de 6–7 tokens
sin contexto son el escenario más difícil para un modelo BERT-base monolingüe español (110M parámetros), que necesita
contexto bidireccional para desambiguar. Un F1 bajo aquí indica que el modelo
depende del contexto para funcionar bien.

**WikiNEuRal** es la versión de mayor calidad del benchmark Wikipedia. Al tener
cuatro veces más entidades que WikiANN, los resultados son estadísticamente más
fiables. La diferencia entre WikiANN y WikiNEuRal refleja el impacto de la
calidad de la anotación: etiquetado ruidoso vs. etiquetado híbrido de alta calidad.

**MultiNERD** es el benchmark más exigente: combina Wikipedia y WikiNoticias
con 15 categorías de entidades (colapsadas en 4 para la evaluación). El resultado
aquí es el indicador más realista del rendimiento en texto variado en producción.

### Implicaciones para la app de filtrado jurídico

El texto jurídico (documentos legales, expedientes, resoluciones) es más parecido
al dominio de CoNLL-2002 (texto formal estructurado) que al de WikiANN (frases cortas
sin contexto). Sin embargo, los documentos jurídicos contienen entidades propias
(nombres de leyes, órganos administrativos, artículos) que ningún benchmark cubre.

El análisis de FP y FN proporciona el inventario concreto de errores que el
pipeline de post-procesado debe corregir para la app.

## Nota metodológica para el paper

Los resultados se reportan como **F1 = F̄ ± σ** donde:
- **F̄** es la media de los F1 individuales de cada frase
- **σ** es la desviación típica muestral de esos F1 por frase
- El **IC95** = F̄ ± 1.96·σ/√n se obtiene por el TCL (n >> 30 en todos los corpus)

El Teorema Central del Límite garantiza que la distribución de μ converge
a una normal para B suficientemente grande. Con B=1000 la convergencia
es prácticamente completa para corpus de ≥200 frases.

**Para reproducir exactamente estos resultados:**
```python
f1_por_frase = [exact_metrics(s["entities"], predict(s))["F1"]
                for s in corpus]
ic = ic95_analitico(f1_por_frase)
# IC95 = ic["mean"] ± 1.96 * ic["se"]
```
El IC95 analítico es **completamente determinista** — dado el mismo corpus
y modelo, cualquier lector obtiene exactamente los mismos valores sin
necesidad de fijar ninguna semilla aleatoria.

**Por qué se cambió C3 (CoNLL-2002 testa → WikiNEuRal-ES):**
CoNLL-2002 testa y testb comparten dominio y fuente (agencia EFE), por lo que
su uso conjunto en el mismo paper no aporta variabilidad de dominio.
WikiNEuRal-ES (Babelscape, 2021) es un benchmark independiente de dominio
Wikipedia con anotación de mayor calidad que WikiANN (generada con BabelNet),
permitiendo comparar el efecto de la calidad de anotación en el F1 observado.

### Notas específicas de BETO-NER

BETO es el BERT oficial para español, pre-entrenado por la Universidad de Chile. Al ser monolingüe (solo español), no reparte capacidad entre idiomas como mBERT. El checkpoint de BETO-NER requiere `use_fast=False` en el tokenizador: sin este parámetro, los offsets de carácter devueltos por el pipeline no corresponden al texto original, lo que hace que el cálculo de exact_metrics sea incorrecto.
